# Global Migration Corridors — A Gravity Model Analysis

**Research question:** Do global migration corridors follow colonial/historical ties,
economic pull, or geographic proximity most strongly?

**Status:** Step 1 (World Bank enrichment data) is live and runnable. Steps 2–4
(UN DESA, CEPII GeoDist, and the merges that depend on them) are stubbed —
fill them in once those files are uploaded.

**Pipeline stages** (see project handoff docs):
1. Load & inspect World Bank enrichment data (imputed + interpolated) — *runnable now*
2. Source UN DESA bilateral migrant stock + CEPII GeoDist — *waiting on files*
3. Build ISO3 crosswalk across all sources, validate overlap — *WB half runnable now*
4. Wrangle UN wide-format Excel → long format — *stub*
5. Merge: UN ⨝ CEPII (on country-pair), UN ⨝ World Bank ×2 (origin + destination) — *stub*
6. Time-period alignment decision (5-yr UN snapshots vs. annual WB data) — *stub*
7. Model: log-log gravity regression + multicollinearity (VIF) check — *stub*
8. Visualize + document limitations honestly — *stub*

**Working principles carried forward from the FIFA project:** validate before
trusting any "clean" join result; document manual corrections explicitly;
report effect sizes, not just p-values; never silently drop or smooth over
data-boundary disagreements between sources.


## 0. Setup

In [3]:
import pandas as pd
import numpy as np
import zipfile
import os

# pip install pycountry  (uncomment if not already installed in your environment)
# !pip install pycountry --quiet
import pycountry

pd.set_option('display.max_columns', 50)


In [4]:
# --- CONFIG: file locations ---
# All files live in the same folder as this notebook.
WB_ZIP_PATH = "Global Socio-Economic & Demograpic.zip"
DATA_DIR = "data"

os.makedirs(DATA_DIR, exist_ok=True)
with zipfile.ZipFile(WB_ZIP_PATH, "r") as z:
    z.extractall(DATA_DIR)

WB_IMPUTED_PATH = os.path.join(DATA_DIR, "world_development_data_imputed.csv")
WB_INTERPOLATED_PATH = os.path.join(DATA_DIR, "world_development_data_interpolated.csv")

UN_DESA_PATH = "undesa_pd_2024_ims_stock_by_sex_destination_and_origin.xlsx"
CEPII_PATH = "dist_cepii.xls"


## 1. Load & inspect World Bank enrichment data

Two files, per the handoff:
- **imputed**: 202 countries, 2000–2021, 26 cols, zero missing values (kNN-imputed + interpolated) — use as the **primary** source per project decision.
- **interpolated**: more columns (33), longer range (1973–2021), ~58,500 missing values — use **selectively**, only for metrics not present in `imputed` (e.g. `MilExp%GDP`, `TaxRevenue%GDP`, `DomCredit%GDP`, `SchEnrollPrim%`, `GrossCapForm%GDP`, `RevenueExGrants%GDP`, `IntermRegion`).


In [6]:
wb_imputed = pd.read_csv(WB_IMPUTED_PATH)
wb_interp = pd.read_csv(WB_INTERPOLATED_PATH)

print("IMPUTED:", wb_imputed.shape)
print("INTERPOLATED:", wb_interp.shape)

extra_cols = sorted(set(wb_interp.columns) - set(wb_imputed.columns))
print("\nColumns only in interpolated (candidates for selective merge-in):")
print(extra_cols)


IMPUTED: (4444, 26)
INTERPOLATED: (9947, 33)

Columns only in interpolated (candidates for selective merge-in):
['DomCredit%GDP', 'GrossCapForm%GDP', 'IntermRegion', 'MilExp%GDP', 'RevenueExGrants%GDP', 'SchEnrollPrim%', 'TaxRevenue%GDP']


In [7]:
# Sanity checks before trusting either file
print("Imputed — nulls per column (should all be 0):")
print(wb_imputed.isna().sum()[wb_imputed.isna().sum() > 0])

print("\nInterpolated — % missing (top 10 worst columns):")
print((wb_interp.isna().mean().sort_values(ascending=False) * 100).head(10).round(1))

print("\nYear range — imputed:", wb_imputed['Year'].min(), '-', wb_imputed['Year'].max())
print("Year range — interpolated:", wb_interp['Year'].min(), '-', wb_interp['Year'].max())

print("\nUnique countries — imputed:", wb_imputed['Country'].nunique())
print("Unique countries — interpolated:", wb_interp['Country'].nunique())


Imputed — nulls per column (should all be 0):
Series([], dtype: int64)

Interpolated — % missing (top 10 worst columns):
DomCredit%GDP          88.1
RevenueExGrants%GDP    58.0
IntermRegion           57.6
TaxRevenue%GDP         57.5
MilExp%GDP             35.3
GrossCapForm%GDP       27.8
Imports%GDP            25.3
Exports%GDP            25.3
IndValAdd%GDP          25.2
AgriValAdd%GDP         24.6
dtype: float64

Year range — imputed: 2000.0 - 2021.0
Year range — interpolated: 1973 - 2021

Unique countries — imputed: 202
Unique countries — interpolated: 203


In [8]:
# Build the selective interpolated-only slice: extra metrics, keyed on Country+Year,
# for years where wb_imputed also has coverage. Merge this into the primary frame later.
wb_interp_extra = wb_interp[['Country', 'Year'] + extra_cols].copy()
wb_interp_extra.head()


,Country,Year,DomCredit%GDP,GrossCapForm%GDP,IntermRegion,MilExp%GDP,RevenueExGrants%GDP,SchEnrollPrim%,TaxRevenue%GDP
0,Afghanistan,1973,NaN,7.307692,NaN,1.868910,NaN,35.214371,NaN
1,Netherlands,1973,NaN,25.746197,NaN,2.702997,38.294098,100.047302,22.460626
2,Poland,1973,NaN,NaN,NaN,NaN,NaN,105.338722,NaN
3,"Egypt, Arab Rep.",1973,NaN,13.197398,NaN,13.513514,NaN,70.221947,NaN
4,Gabon,1973,NaN,37.554314,Middle Africa,1.303538,NaN,130.283417,NaN


### 1c. Missingness audit — is each extra column worth keeping?

Two separate questions, easy to conflate:
1. **Does restricting to the years `gravity_df` actually uses (2000, 2005, 2010, 2015,
   2020, 2021 — the only years the Step 5 nearest-year mapping ever looks up)
   change the missingness picture?**
2. **Is missingness scattered across countries/years, or concentrated in specific
   countries (entirely absent, every year)?** The latter is MNAR (missing *not*
   at random) — those countries likely share something in common (reporting
   capacity, economic structure), so dropping or imputing risks quietly biasing
   the sample toward better-reporting countries. Don't impute or drop rows over
   this without knowing which case it is.


In [10]:
# Hardcoded from Step 5's nearest-year mapping (World Bank's own coverage: 2000-2021,
# nearest-matched to UN's 1990/1995/2000/2005/2010/2015/2020/2024 snapshots).
relevant_years = [2000, 2005, 2010, 2015, 2020, 2021]

print("Missingness -- full file (1973-2021):")
print((wb_interp[extra_cols].isna().mean() * 100).round(1))

print("\nMissingness -- restricted to years gravity_df actually uses:")
relevant = wb_interp[wb_interp['Year'].isin(relevant_years)]
print((relevant[extra_cols].isna().mean() * 100).round(1))


Missingness -- full file (1973-2021):
DomCredit%GDP          88.1
GrossCapForm%GDP       27.8
IntermRegion           57.6
MilExp%GDP             35.3
RevenueExGrants%GDP    58.0
SchEnrollPrim%         20.3
TaxRevenue%GDP         57.5
dtype: float64

Missingness -- restricted to years gravity_df actually uses:
DomCredit%GDP          82.0
GrossCapForm%GDP       17.5
IntermRegion           57.6
MilExp%GDP             27.2
RevenueExGrants%GDP    44.3
SchEnrollPrim%         17.7
TaxRevenue%GDP         43.8
dtype: float64


In [11]:
# Country-concentration check: is a column missing for SOME years per country (scattered,
# safer) or ENTIRELY missing for whole countries (structural non-reporting, MNAR risk)?
by_country = relevant.groupby('Country')[extra_cols].apply(lambda g: g.isna().mean())

print(f"{'Column':<22}{'Entirely missing (all yrs)':<30}{'Entirely present':<20}")
for col in extra_cols:
    all_missing = (by_country[col] == 1.0).sum()
    all_present = (by_country[col] == 0.0).sum()
    n = len(by_country)
    print(f"{col:<22}{all_missing}/{n}{'':<24}{all_present}/{n}")


Column                Entirely missing (all yrs)    Entirely present    
DomCredit%GDP         144/203                        6/203
GrossCapForm%GDP      21/203                        147/203
IntermRegion          117/203                        86/203
MilExp%GDP            44/203                        126/203
RevenueExGrants%GDP   54/203                        59/203
SchEnrollPrim%        6/203                        117/203
TaxRevenue%GDP        53/203                        61/203


**Decision, per column (documented, not silently applied):**

| Column | Verdict | Reason |
|---|---|---|
| `IntermRegion` | **Drop entirely** | Not real missingness — a categorical field only populated for certain UN sub-regions; `NaN` means "not applicable," not "unknown." Redundant with `SubRegion`, already complete in `wb_imputed`. |
| `DomCredit%GDP` | **Drop entirely** | ~71% of countries never report this at all, in any year — structural non-reporting (MNAR), not a fillable gap. |
| `RevenueExGrants%GDP`, `TaxRevenue%GDP` | **Keep out of primary model; robustness-check only if used** | ~26–27% of countries entirely missing — a real but smaller version of the same MNAR risk. |
| `MilExp%GDP` | **Usable with caution** | ~22% of countries entirely missing — borderline; fine as a secondary variable, not central to the hypothesis. |
| `GrossCapForm%GDP`, `SchEnrollPrim%` | **Keep** | Only ~3–10% of countries entirely missing — low enough for standard listwise handling in the regression. |


In [13]:
columns_to_drop = ['IntermRegion', 'DomCredit%GDP']
wb_interp_extra = wb_interp_extra.drop(columns=columns_to_drop)
print("wb_interp_extra columns after drop:", list(wb_interp_extra.columns))


wb_interp_extra columns after drop: ['Country', 'Year', 'GrossCapForm%GDP', 'MilExp%GDP', 'RevenueExGrants%GDP', 'SchEnrollPrim%', 'TaxRevenue%GDP']


### 1d. ISO3 crosswalk — World Bank side

Per the enrichment reference doc: join on **ISO3 codes**, not raw country name
strings, since UN DESA / CEPII / World Bank each name countries differently.
Doing the World Bank half now since that data is already in hand; the UN and
CEPII halves get added in Step 3 once those files exist.


In [15]:
def name_to_iso3(name):
    """Best-effort match of a country name string to its ISO alpha-3 code."""
    try:
        return pycountry.countries.lookup(name).alpha_3
    except LookupError:
        return None  # flag for manual review -- do NOT silently drop

wb_countries = pd.Series(wb_imputed['Country'].unique(), name='Country').to_frame()
wb_countries['iso3'] = wb_countries['Country'].apply(name_to_iso3)

unmatched_wb = wb_countries[wb_countries['iso3'].isna()]['Country'].tolist()
print(f"{len(unmatched_wb)} / {len(wb_countries)} World Bank country names could not be auto-matched:")
print(unmatched_wb)


20 / 202 World Bank country names could not be auto-matched:
['Micronesia, Fed. Sts.', 'Lao PDR', 'Macao SAR, China', 'Venezuela, RB', 'Bahamas, The', 'Korea, Rep.', 'Egypt, Arab Rep.', 'Yemen, Rep.', 'Congo, Dem. Rep.', "Cote d'Ivoire", 'Curacao', 'Congo, Rep.', 'West Bank and Gaza', 'Iran, Islamic Rep.', 'Hong Kong SAR, China', 'Gambia, The', 'St. Lucia', 'Turkiye', 'St. Kitts and Nevis', 'St. Vincent and the Grenadines']


In [16]:
# Manual overrides — add one entry per name in `unmatched_wb` above, verified individually.
# This dict is intentionally seeded with the classic World Bank long-form cases;
# re-run the cell above after populating and confirm unmatched_wb empties out.
manual_iso3_overrides_wb = {
    'Bahamas, The': 'BHS',
    'Congo, Dem. Rep.': 'COD',
    'Congo, Rep.': 'COG',
    "Cote d'Ivoire": 'CIV',
    'Curacao': 'CUW',
    'Egypt, Arab Rep.': 'EGY',
    'Gambia, The': 'GMB',
    'Hong Kong SAR, China': 'HKG',
    'Iran, Islamic Rep.': 'IRN',
    'Korea, Rep.': 'KOR',
    'Lao PDR': 'LAO',
    'Macao SAR, China': 'MAC',
    'Micronesia, Fed. Sts.': 'FSM',
    'St. Kitts and Nevis': 'KNA',
    'St. Lucia': 'LCA',
    'St. Vincent and the Grenadines': 'VCT',
    'Turkiye': 'TUR',
    'Venezuela, RB': 'VEN',
    'West Bank and Gaza': 'PSE',  # not a UN member state; ISO uses PSE (\"State of Palestine\") -- document this choice
    'Yemen, Rep.': 'YEM',
}

wb_countries['iso3'] = wb_countries['iso3'].fillna(wb_countries['Country'].map(manual_iso3_overrides_wb))

still_unmatched = wb_countries[wb_countries['iso3'].isna()]
print(f"Still unmatched after manual overrides: {len(still_unmatched)}")
still_unmatched


Still unmatched after manual overrides: 0


,Country,iso3


In [17]:
wb_imputed = wb_imputed.merge(wb_countries, on='Country', how='left')
wb_interp_extra = wb_interp_extra.merge(wb_countries, on='Country', how='left')

print("wb_imputed rows with no iso3 (should be 0 or explicitly known):", wb_imputed['iso3'].isna().sum())
wb_imputed.head(3)


wb_imputed rows with no iso3 (should be 0 or explicitly known): 0


,Year,Country,Region,SubRegion,SurfAreaSqKm,PopTotal,PopDens,PopGrowth%,GDP,GDPGrowth%,AdolFertRate,AgriValAdd%GDP,Exports%GDP,FertRate,FDINetBoP,GNI/CapAtlas,GNIAtlas,Imports%GDP,IndValAdd%GDP,InflConsPric%,LifeExpBirth,MerchTrade%GDP,MobileSubs/100,MortRateU5,NetMigr,UrbanPopGrowth%,iso3
0,2000.0,Afghanistan,Asia,Southern Asia,652860.0,19542982.0,29.963329,1.443803,1.801248e+10,-5.206288,152.572,33.096776,13.315247,7.534,1.700000e+05,916.923281,1.778669e+10,41.312634,17.178775,37.611028,55.298000,52.777048,0.000000,129.3,-1007135.0,1.861377,AFG
1,2000.0,Malta,Europe,Southern Europe,320.0,390087.0,1219.021875,0.645267,4.323339e+09,19.681791,19.869,1.871997,120.247484,1.680,7.431853e+08,10950.000000,4.273280e+09,128.374581,26.939987,-4.512396,78.348780,135.682159,28.667475,7.6,1799.0,0.952299,MLT
2,2000.0,Belgium,Europe,Western Europe,30530.0,10251250.0,338.548547,0.242518,2.367925e+11,3.716679,11.915,1.176005,72.547395,1.670,8.873871e+10,25890.000000,2.654129e+11,69.682740,24.943833,2.014617,77.721951,154.515900,54.840339,5.9,32262.0,0.308431,BEL


## 2. Load UN DESA + CEPII GeoDist

**UN DESA** (`International Migrant Stock 2024: Destination and origin`, Table 1):
wide Excel, header spans multiple merged rows — real header is on row 11 (index 10),
data starts row 12 (index 11). Rows mix **country-level** and **region/aggregate**
entries (World, Sub-Saharan Africa, income groups, etc.) in the same columns —
verified below that `Location code of destination/origin < 900` cleanly separates
real countries (UN M49 numeric codes) from aggregates (codes >= 900).

**CEPII GeoDist** (`dist_cepii.xls`): single sheet, already keyed on ISO alpha-3
(`iso_o`, `iso_d`) — no name-matching needed for this source.


In [19]:
xl = pd.ExcelFile(UN_DESA_PATH)
print("UN DESA sheets:", xl.sheet_names)

un_raw = xl.parse("Table 1", header=10)
print("Raw shape:", un_raw.shape)
un_raw.columns = [str(c) for c in un_raw.columns]
print(list(un_raw.columns))


UN DESA sheets: ['Table of contents', 'Table 1', 'Table 2', 'Table 3', 'Table 4', 'Table 5', 'Table 6', 'Table 7', 'Table 8', 'Table 9', 'Migrant notes']
Raw shape: (28030, 31)
['Index', 'Region, development group, country or area of destination', 'Coverage', 'Data type', 'Location code of destination', 'Region, development group, country or area of origin', 'Location code of origin', '1990', '1995', '2000', '2005', '2010', '2015', '2020', '2024', '1990.1', '1995.1', '2000.1', '2005.1', '2010.1', '2015.1', '2020.1', '2024.1', '1990.2', '1995.2', '2000.2', '2005.2', '2010.2', '2015.2', '2020.2', '2024.2']


In [20]:
# Verify the code-900 boundary before trusting the filter (don't assume, check)
code_check = un_raw[['Region, development group, country or area of destination',
                      'Location code of destination']].drop_duplicates().sort_values('Location code of destination')
print("Rows just below 900 (should be real countries):")
print(code_check[(code_check['Location code of destination'] >= 850) & (code_check['Location code of destination'] < 900)].to_string())
print("\nRows at/above 900 (should be aggregates):")
print(code_check[code_check['Location code of destination'] >= 900].head(10).to_string())


Rows just below 900 (should be real countries):
      Region, development group, country or area of destination  Location code of destination
22869                             United States Virgin Islands*                           850
9184                                               Burkina Faso                           854
25364                                                   Uruguay                           858
10670                                                Uzbekistan                           860
25405                        Venezuela (Bolivarian Republic of)                           862
28002                                Wallis and Futuna Islands*                           876
27871                                                     Samoa                           882
13808                                                     Yemen                           887
7097                                                     Zambia                           894

Rows at/abo

In [21]:
# Filter to genuine country-to-country pairs only
un_countries = un_raw[
    (un_raw['Location code of destination'] < 900) &
    (un_raw['Location code of origin'] < 900)
].copy()

print("Country-to-country rows:", un_countries.shape)
print("Unique destination countries:", un_countries['Region, development group, country or area of destination'].nunique())
print("Unique origin countries:", un_countries['Region, development group, country or area of origin'].nunique())


Country-to-country rows: (9348, 31)
Unique destination countries: 231
Unique origin countries: 233


In [22]:
cepii_raw = pd.read_excel(CEPII_PATH)
print(cepii_raw.shape)
print(list(cepii_raw.columns))
print("Unique origin countries in CEPII:", cepii_raw['iso_o'].nunique())
cepii_raw.head(3)


(50176, 14)
['iso_o', 'iso_d', 'contig', 'comlang_off', 'comlang_ethno', 'colony', 'comcol', 'curcol', 'col45', 'smctry', 'dist', 'distcap', 'distw', 'distwces']
Unique origin countries in CEPII: 224


,iso_o,iso_d,contig,comlang_off,comlang_ethno,colony,comcol,curcol,col45,smctry,dist,distcap,distw,distwces
0,ABW,ABW,0,0,0,0,0,0,0,0,5.225315,5.225315,25.09354,23.04723
1,ABW,AFG,0,0,0,0,0,0,0,0,13257.810000,13257.810000,13168.22,13166.37
2,ABW,AGO,0,0,0,0,0,0,0,0,9516.913000,9516.913000,9587.316,9584.193


### 2b. CEPII legacy ISO3 codes

`dist_cepii.xls` was last updated in 2004 and predates several ISO code changes.
Confirmed by inspection (Step 3's overlap diff) — these are clean renamings, safe
to remap. Two others (`YUG` former Yugoslavia, `ANT` Netherlands Antilles) are
**genuinely obsolete political entities with no current equivalent** — left
unmapped and documented rather than force-mapped to a country that didn't exist
when CEPII compiled the data.


In [24]:
cepii_legacy_code_fixes = {
    'ROM': 'ROU',   # Romania
    'ZAR': 'COD',   # DR Congo
    'TMP': 'TLS',   # Timor-Leste
    'PAL': 'PSE',   # Palestine
    # 'YUG' (former Yugoslavia) and 'ANT' (Netherlands Antilles, dissolved 2010)
    # intentionally left unmapped -- no current-day ISO3 equivalent exists.
}

cepii_raw['iso_o'] = cepii_raw['iso_o'].replace(cepii_legacy_code_fixes)
cepii_raw['iso_d'] = cepii_raw['iso_d'].replace(cepii_legacy_code_fixes)

print("CEPII origin codes after fix:", cepii_raw['iso_o'].nunique())


CEPII origin codes after fix: 224


### 2c. Fix `distw`/`distwces` dtype

CEPII uses a period (`.`) as its missing-value placeholder (a Stata convention) --
2,215 rows have `'.'` instead of a number in `distw`, which forced pandas to store
the entire column as text (`object`) rather than `float64`. Converting explicitly
turns those placeholders into proper `NaN` rather than leaving a silently-broken
text column. Not used in the current PPML model (which uses `dist`), but fixed
here so it's usable if a future analysis wants the population-weighted distance
measure instead.


In [26]:
cepii_raw['distw'] = pd.to_numeric(cepii_raw['distw'], errors='coerce')
cepii_raw['distwces'] = pd.to_numeric(cepii_raw['distwces'], errors='coerce')

print("distw dtype:", cepii_raw['distw'].dtype, "-- missing after conversion:", cepii_raw['distw'].isna().sum())
print("distwces dtype:", cepii_raw['distwces'].dtype, "-- missing after conversion:", cepii_raw['distwces'].isna().sum())


distw dtype: float64 -- missing after conversion: 2215
distwces dtype: float64 -- missing after conversion: 2215


## 3. Full ISO3 crosswalk + validation (all three sources)

CEPII already keys on ISO alpha-3 (`iso_o`/`iso_d`) — no name-matching needed there.
UN DESA needs the same `name_to_iso3` + manual-override pattern used for World
Bank in Step 1b. Some UN names carry a trailing `*` (footnote marker) that must
be stripped before matching.


In [28]:
un_dest_names = pd.Series(
    un_countries['Region, development group, country or area of destination'].str.rstrip('*').unique(),
    name='country_name'
).to_frame()
un_dest_names['iso3'] = un_dest_names['country_name'].apply(name_to_iso3)

unmatched_un = un_dest_names[un_dest_names['iso3'].isna()]['country_name'].tolist()
print(f"{len(unmatched_un)} / {len(un_dest_names)} UN country names could not be auto-matched:")
print(unmatched_un)


15 / 231 UN country names could not be auto-matched:
['Democratic Republic of the Congo', 'Saint Helena', 'China, Hong Kong SAR', 'China, Macao SAR', 'China, Taiwan Province of China', "Dem. People's Republic of Korea", 'Republic of Korea', 'Iran (Islamic Republic of)', 'State of Palestine', 'Channel Islands', 'United States Virgin Islands', 'Bolivia (Plurinational State of)', 'Venezuela (Bolivarian Republic of)', 'Micronesia (Fed. States of)', 'Wallis and Futuna Islands']


In [29]:
# Manual overrides, verified individually against each name in `unmatched_un` above.
manual_iso3_overrides_un = {
    'Bolivia (Plurinational State of)': 'BOL',
    'Channel Islands': None,                      # not a single ISO country (Jersey+Guernsey combined) -- document, leave unmapped
    'China, Hong Kong SAR': 'HKG',
    'China, Macao SAR': 'MAC',
    'China, Taiwan Province of China': 'TWN',
    "Dem. People's Republic of Korea": 'PRK',
    'Democratic Republic of the Congo': 'COD',
    'Iran (Islamic Republic of)': 'IRN',
    'Micronesia (Fed. States of)': 'FSM',
    'Republic of Korea': 'KOR',
    'Saint Helena': 'SHN',
    'State of Palestine': 'PSE',
    'United States Virgin Islands': 'VIR',
    'Venezuela (Bolivarian Republic of)': 'VEN',
    'Wallis and Futuna Islands': 'WLF',
}

un_dest_names['iso3'] = un_dest_names['iso3'].fillna(un_dest_names['country_name'].map(manual_iso3_overrides_un))

still_unmatched_un = un_dest_names[un_dest_names['iso3'].isna()]
print(f"Still unmatched after manual overrides: {len(still_unmatched_un)}")
print(still_unmatched_un)


Still unmatched after manual overrides: 1
        country_name  iso3
119  Channel Islands  None


In [30]:
# Apply the UN name -> ISO3 map to both destination and origin columns
un_iso3_map = dict(zip(un_dest_names['country_name'], un_dest_names['iso3']))

un_countries['destination_iso3'] = un_countries['Region, development group, country or area of destination'].str.rstrip('*').map(un_iso3_map)
un_countries['origin_iso3'] = un_countries['Region, development group, country or area of origin'].str.rstrip('*').map(un_iso3_map)

print("Rows with no destination_iso3:", un_countries['destination_iso3'].isna().sum())
print("Rows with no origin_iso3:", un_countries['origin_iso3'].isna().sum())
# 'Channel Islands' rows will show up here as expected (documented above, not silently dropped)


Rows with no destination_iso3: 4
Rows with no origin_iso3: 20


In [31]:
# Overlap validation across all three sources -- inspect before trusting any merge
un_codes = set(un_countries['destination_iso3'].dropna()) | set(un_countries['origin_iso3'].dropna())
cepii_codes = set(cepii_raw['iso_o'].dropna()) | set(cepii_raw['iso_d'].dropna())
wb_codes = set(wb_imputed['iso3'].dropna())

print(f"UN codes: {len(un_codes)}")
print(f"CEPII codes: {len(cepii_codes)}")
print(f"World Bank codes: {len(wb_codes)}")
print()
print("In UN but not CEPII:", sorted(un_codes - cepii_codes))
print("In CEPII but not UN:", sorted(cepii_codes - un_codes))
print("In UN but not World Bank:", sorted(un_codes - wb_codes))
# --> Every diff needs a decision: real naming gap to fix, or a country genuinely
#     absent from one source (e.g. small territories CEPII doesn't cover).
#     Document whichever it is -- do not silently drop.


UN codes: 230
CEPII codes: 224
World Bank codes: 202

In UN but not CEPII: ['ASM', 'CUW', 'GUM', 'IMN', 'LIE', 'MCO', 'MNE', 'MYT', 'SRB', 'SSD', 'SXM', 'VIR']
In CEPII but not UN: ['ANT', 'CCK', 'CXR', 'NFK', 'PCN', 'YUG']
In UN but not World Bank: ['AIA', 'ASM', 'COK', 'ESH', 'FLK', 'GIB', 'GLP', 'GUF', 'GUM', 'IMN', 'MCO', 'MNP', 'MSR', 'MTQ', 'MYT', 'NIU', 'PRK', 'REU', 'SHN', 'SMR', 'SPM', 'SSD', 'SXM', 'TKL', 'TWN', 'VGB', 'VIR', 'WLF']


## 4. Wrangle UN DESA wide format → long format

Target shape: one row per (origin_iso3, destination_iso3, year, migrant_stock).
The "both sexes" totals live in the plain-year columns (1990, 1995, ..., 2024);
male/female breakdowns live in the `.1`/`.2` suffixed columns pandas generated
from the repeated year headers. Gravity model needs totals — keep sex breakdown
available but build the primary long table off "both sexes."


In [33]:
year_cols_total = ['1990', '1995', '2000', '2005', '2010', '2015', '2020', '2024']

un_long = un_countries.melt(
    id_vars=['destination_iso3', 'origin_iso3'],
    value_vars=year_cols_total,
    var_name='year',
    value_name='migrant_stock'
)
un_long['year'] = un_long['year'].astype(int)

# Drop rows with no reported stock (not the same as zero -- UN leaves these blank when unreported)
before = len(un_long)
un_long = un_long.dropna(subset=['migrant_stock'])
print(f"Dropped {before - len(un_long)} rows with no reported stock (unreported, not zero -- documented, not silently zero-filled)")
print("un_long shape:", un_long.shape)
un_long.head()


Dropped 1 rows with no reported stock (unreported, not zero -- documented, not silently zero-filled)
un_long shape: (74783, 4)


,destination_iso3,origin_iso3,year,migrant_stock
0,BDI,KEN,1990,186.0
1,BDI,RWA,1990,221943.0
2,BDI,UGA,1990,1833.0
3,BDI,TZA,1990,11912.0
4,BDI,COD,1990,36654.0


## 5. Time-period alignment decision

UN migrant stock: 5-year snapshots (1990–2024). World Bank: annual, but only
2000–2021. Three UN snapshot years (1990, 1995, 2024) fall **outside** World
Bank's range entirely, so an exact-year join would silently lose those rows'
economic data.

**Decision made here:** snap World Bank annual data to the *nearest available*
year for each UN snapshot, rather than requiring an exact match. This is the
"simpler, some info loss" option from the original tradeoff — chosen because
interpolating UN's 5-year stock data to annual would fabricate migration trend
assumptions the source data doesn't support, which is a bigger methodological
risk than a few years of GDP staleness. **This is a real modeling choice — state
it plainly in any writeup**, including which UN years borrow a nearby WB year's
GDP rather than their own.


In [35]:
wb_years = sorted(wb_imputed['Year'].unique())
un_snapshot_years = sorted(un_long['year'].unique())

def nearest_wb_year(year):
    return min(wb_years, key=lambda y: abs(y - year))

year_map = {y: nearest_wb_year(y) for y in un_snapshot_years}
print("UN snapshot year -> nearest available World Bank year:")
for un_y, wb_y in year_map.items():
    flag = "  <-- outside WB range, using nearest" if un_y not in wb_years else ""
    print(f"  {un_y} -> {wb_y}{flag}")


UN snapshot year -> nearest available World Bank year:
  1990 -> 2000.0  <-- outside WB range, using nearest
  1995 -> 2000.0  <-- outside WB range, using nearest
  2000 -> 2000.0
  2005 -> 2005.0
  2010 -> 2010.0
  2015 -> 2015.0
  2020 -> 2020.0
  2024 -> 2021.0  <-- outside WB range, using nearest


In [36]:
un_long['wb_lookup_year'] = un_long['year'].map(year_map)
un_long.head()


,destination_iso3,origin_iso3,year,migrant_stock,wb_lookup_year
0,BDI,KEN,1990,186.0,2000.0
1,BDI,RWA,1990,221943.0,2000.0
2,BDI,UGA,1990,1833.0,2000.0
3,BDI,TZA,1990,11912.0,2000.0
4,BDI,COD,1990,36654.0,2000.0


## 6. Merge pipeline

```
UN (origin_iso3, destination_iso3, year, wb_lookup_year, migrant_stock)
   ├── merge on (origin_iso3, destination_iso3)          → CEPII (distance, colonial_tie, language, ...)
   ├── merge on (origin_iso3, wb_lookup_year)             → World Bank imputed (origin GDP, pop, ...)
   ├── merge on (destination_iso3, wb_lookup_year)        → World Bank imputed (destination GDP, pop, ...) [suffix _dest]
   ├── merge on (origin_iso3, wb_lookup_year)             → World Bank interpolated extras (origin), post Step 1c cleanup
   └── merge on (destination_iso3, wb_lookup_year)        → World Bank interpolated extras (destination)
```
Per the enrichment reference doc, World Bank data gets merged in **twice** —
gravity models need economic pull on both sides of the corridor. Joining on
`wb_lookup_year` (Step 5's nearest-year mapping) rather than the raw UN `year`
is what fixes the ~40% GDP-match gap. The interpolated-file extras (`GrossCapForm%GDP`,
`MilExp%GDP`, `RevenueExGrants%GDP`, `SchEnrollPrim%`, `TaxRevenue%GDP`) are merged
in too, post-cleanup — `IntermRegion` and `DomCredit%GDP` were already dropped in
Step 1c and never reach this merge.


In [38]:
cepii_slim = cepii_raw.rename(columns={'iso_o': 'origin_iso3', 'iso_d': 'destination_iso3'})

wb_origin = wb_imputed.add_suffix('_origin').rename(columns={'iso3_origin': 'origin_iso3', 'Year_origin': 'wb_lookup_year'})
wb_dest = wb_imputed.add_suffix('_dest').rename(columns={'iso3_dest': 'destination_iso3', 'Year_dest': 'wb_lookup_year'})

wb_extra_origin = wb_interp_extra.add_suffix('_origin').rename(columns={'iso3_origin': 'origin_iso3', 'Year_origin': 'wb_lookup_year'})
wb_extra_dest = wb_interp_extra.add_suffix('_dest').rename(columns={'iso3_dest': 'destination_iso3', 'Year_dest': 'wb_lookup_year'})

gravity_df = (
    un_long
    .merge(cepii_slim, on=['origin_iso3', 'destination_iso3'], how='left')
    .merge(wb_origin, on=['origin_iso3', 'wb_lookup_year'], how='left')
    .merge(wb_dest, on=['destination_iso3', 'wb_lookup_year'], how='left')
    .merge(wb_extra_origin, on=['origin_iso3', 'wb_lookup_year'], how='left')
    .merge(wb_extra_dest, on=['destination_iso3', 'wb_lookup_year'], how='left')
)

print("gravity_df shape:", gravity_df.shape)
print("\nMerge coverage check -- rows with no CEPII match:", gravity_df['dist'].isna().sum())
print("Rows with no origin GDP match:", gravity_df['GDP_origin'].isna().sum())
print("Rows with no destination GDP match:", gravity_df['GDP_dest'].isna().sum())
print("Rows with no origin GrossCapForm%GDP match:", gravity_df['GrossCapForm%GDP_origin'].isna().sum())
gravity_df.head()


gravity_df shape: (74783, 79)

Merge coverage check -- rows with no CEPII match: 4232
Rows with no origin GDP match: 2528
Rows with no destination GDP match: 2528
Rows with no origin GrossCapForm%GDP match: 10054


,destination_iso3,origin_iso3,year,migrant_stock,wb_lookup_year,contig,comlang_off,comlang_ethno,colony,comcol,curcol,col45,smctry,dist,distcap,distw,distwces,Country_origin_x,Region_origin,SubRegion_origin,SurfAreaSqKm_origin,PopTotal_origin,PopDens_origin,PopGrowth%_origin,GDP_origin,...,FertRate_dest,FDINetBoP_dest,GNI/CapAtlas_dest,GNIAtlas_dest,Imports%GDP_dest,IndValAdd%GDP_dest,InflConsPric%_dest,LifeExpBirth_dest,MerchTrade%GDP_dest,MobileSubs/100_dest,MortRateU5_dest,NetMigr_dest,UrbanPopGrowth%_dest,Country_origin_y,GrossCapForm%GDP_origin,MilExp%GDP_origin,RevenueExGrants%GDP_origin,SchEnrollPrim%_origin,TaxRevenue%GDP_origin,Country_dest_y,GrossCapForm%GDP_dest,MilExp%GDP_dest,RevenueExGrants%GDP_dest,SchEnrollPrim%_dest,TaxRevenue%GDP_dest
0,BDI,KEN,1990,186.0,2000.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,867.4281,867.4281,877.5054,852.7606,Kenya,Africa,Sub-Saharan Africa,580370.0,30851606.0,54.207411,2.915447,1.270535e+10,...,6.872,1.168325e+07,140.0,8.593409e+08,16.234754,15.539571,38.944893,47.514,22.745913,0.258733,154.6,-40114.0,4.621538,Kenya,17.414091,1.313742,NaN,94.130013,NaN,Burundi,2.781138,4.862402,NaN,60.719742,NaN
1,BDI,RWA,1990,221943.0,2000.0,1.0,1.0,1.0,0.0,1.0,0.0,0.0,1.0,180.0060,180.0060,162.1818,146.9860,Rwanda,Africa,Sub-Saharan Africa,26340.0,8109989.0,328.738914,1.245731,2.068763e+09,...,6.872,1.168325e+07,140.0,8.593409e+08,16.234754,15.539571,38.944893,47.514,22.745913,0.258733,154.6,-40114.0,4.621538,Rwanda,12.292775,3.535503,NaN,97.399910,NaN,Burundi,2.781138,4.862402,NaN,60.719742,NaN
2,BDI,UGA,1990,1833.0,2000.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,549.2128,549.2128,555.0759,526.8893,Uganda,Africa,Sub-Saharan Africa,241550.0,24020697.0,120.217692,3.135356,6.193247e+09,...,6.872,1.168325e+07,140.0,8.593409e+08,16.234754,15.539571,38.944893,47.514,22.745913,0.258733,154.6,-40114.0,4.621538,Uganda,19.483822,1.764608,NaN,128.961121,NaN,Burundi,2.781138,4.862402,NaN,60.719742,NaN
3,BDI,TZA,1990,11912.0,2000.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,1172.4890,777.6951,893.7339,687.5458,Tanzania,Africa,Sub-Saharan Africa,947300.0,34463704.0,38.906868,2.836808,1.337598e+10,...,6.872,1.168325e+07,140.0,8.593409e+08,16.234754,15.539571,38.944893,47.514,22.745913,0.258733,154.6,-40114.0,4.621538,Tanzania,17.456793,1.487264,NaN,68.970528,NaN,Burundi,2.781138,4.862402,NaN,60.719742,NaN
4,BDI,COD,1990,36654.0,2000.0,1.0,1.0,1.0,0.0,1.0,0.0,0.0,0.0,1559.9810,1559.9810,1209.6550,666.6916,"Congo, Dem. Rep.",Africa,Sub-Saharan Africa,2344860.0,48616317.0,21.444748,2.898841,1.908805e+10,...,6.872,1.168325e+07,140.0,8.593409e+08,16.234754,15.539571,38.944893,47.514,22.745913,0.258733,154.6,-40114.0,4.621538,"Congo, Dem. Rep.",14.433496,0.220261,NaN,NaN,NaN,Burundi,2.781138,4.862402,NaN,60.719742,NaN


## 6b. Country-categorization pass

Step 3's overlap diffs surfaced country codes present in one source but missing
from another. These aren't all the same kind of gap — treating them as one
undifferentiated "missing data" problem risks either silently dropping real
countries (like Taiwan) or wasting effort "fixing" gaps that are correct as-is
(like Yugoslavia, which genuinely no longer exists).

**Step A — build the reference table first, before touching any data.** Six
categories, each with a different correct action:


In [40]:
country_categories = {
    "dissolved_no_successor_code": {
        'YUG': 'Yugoslavia -- dissolved 1992-2006, split into Serbia, Montenegro, Bosnia, Croatia, Slovenia, N. Macedonia',
        'ANT': 'Netherlands Antilles -- dissolved 2010, split into Curacao, Sint Maarten, Bonaire/Saba/St.Eustatius',
    },
    "successor_missing_from_cepii": {
        'SSD': 'South Sudan -- independence 2011',
        'SRB': 'Serbia -- post-2006 split',
        'MNE': 'Montenegro -- post-2006 split',
        'CUW': 'Curacao -- post-2010 split',
        'SXM': 'Sint Maarten -- post-2010 split',
    },
    "microstate_unclear_gap": {
        'LIE': 'Liechtenstein', 'MCO': 'Monaco', 'SMR': 'San Marino',
    },
    "political_non_reporter": {
        'TWN': 'Taiwan -- not a World Bank/UN member state',
        'PRK': 'North Korea -- does not report to World Bank',
    },
    "disputed_territory": {
        'ESH': 'Western Sahara',
    },
    "non_sovereign_territory": {
        'ASM':'American Samoa','GUM':'Guam','IMN':'Isle of Man','MYT':'Mayotte','VIR':'US Virgin Islands',
        'VGB':'British Virgin Islands','AIA':'Anguilla','COK':'Cook Islands','FLK':'Falkland Islands',
        'GIB':'Gibraltar','GLP':'Guadeloupe','GUF':'French Guiana','MNP':'N. Mariana Islands','MSR':'Montserrat',
        'MTQ':'Martinique','NIU':'Niue','REU':'Reunion','SHN':'Saint Helena','SPM':'St. Pierre & Miquelon',
        'TKL':'Tokelau','WLF':'Wallis and Futuna','CCK':'Cocos Islands','CXR':'Christmas Island',
        'NFK':'Norfolk Island','PCN':'Pitcairn',
    },
}

for cat, items in country_categories.items():
    print(f"{cat}: {len(items)} codes")


dissolved_no_successor_code: 2 codes
successor_missing_from_cepii: 5 codes
microstate_unclear_gap: 3 codes
political_non_reporter: 2 codes
disputed_territory: 1 codes
non_sovereign_territory: 25 codes


**Step B — measure real impact, weighted by migrant volume, not code count.**

A category with 25 country codes could still be negligible; a single code
(Taiwan) could outweigh all 25 combined. For each category, check what share
of `gravity_df`'s total `migrant_stock` involves a country from that category,
as either origin or destination.


In [42]:
total_stock = gravity_df['migrant_stock'].sum()
print(f"Total migrant_stock in gravity_df: {total_stock:,.0f}\n")

for cat, items in country_categories.items():
    codes = set(items.keys())
    mask = gravity_df['origin_iso3'].isin(codes) | gravity_df['destination_iso3'].isin(codes)
    cat_stock = gravity_df.loc[mask, 'migrant_stock'].sum()
    pct = 100 * cat_stock / total_stock
    n_rows = mask.sum()
    print(f"{cat:<32} rows={n_rows:<7} stock={cat_stock:>15,.0f}  ({pct:.3f}% of total)")


Total migrant_stock in gravity_df: 1,589,366,186

dissolved_no_successor_code      rows=0       stock=              0  (0.000% of total)
successor_missing_from_cepii     rows=2104    stock=     25,007,035  (1.573% of total)
microstate_unclear_gap           rows=1464    stock=        501,628  (0.032% of total)
political_non_reporter           rows=768     stock=     10,774,971  (0.678% of total)
disputed_territory               rows=120     stock=      1,415,303  (0.089% of total)
non_sovereign_territory          rows=2864    stock=      5,523,994  (0.348% of total)


**Step C — decide and act, per category, using Step B's numbers.**

| Category | % of stock | Decision |
|---|---|---|
| `dissolved_no_successor_code` | 0.000% | No action needed — codes never appear in `gravity_df` at all (UN never reports on them); document only |
| `successor_missing_from_cepii` | 1.573% | **Real gap, worth a future fix.** Rows retained but lack CEPII gravity variables (`dist`/`colony`/`comlang`). Not imputed. Follow-up: `geo_cepii` coordinates could fill distance if sourced later |
| `microstate_unclear_gap` | 0.032% | Negligible — document only |
| `political_non_reporter` | 0.678% | **Real gap (Taiwan + N. Korea).** Rows retained but lack World Bank GDP. Not imputed. Follow-up: external GDP source (IMF) if this corridor matters to your specific analysis |
| `disputed_territory` | 0.089% | Negligible — document only |
| `non_sovereign_territory` | 0.348% | Negligible across 25 codes — document only |

No values are invented or dropped here — this just makes each row's status **explicit and attributable**, instead of letting Step 7's regression silently drop them via `NaN` with no record of why.

In [44]:
code_to_category = {code: cat for cat, items in country_categories.items() for code in items}

gravity_df['origin_category'] = gravity_df['origin_iso3'].map(code_to_category)
gravity_df['destination_category'] = gravity_df['destination_iso3'].map(code_to_category)

flagged = gravity_df['origin_category'].notna() | gravity_df['destination_category'].notna()
print(f"Rows flagged with a documented category: {flagged.sum()} / {len(gravity_df)} ({100*flagged.mean():.2f}%)")
print()
print("By origin_category:")
print(gravity_df['origin_category'].value_counts())
print()
print("By destination_category:")
print(gravity_df['destination_category'].value_counts())

Rows flagged with a documented category: 7208 / 74783 (9.64%)

By origin_category:
origin_category
non_sovereign_territory         1400
successor_missing_from_cepii     880
political_non_reporter           424
microstate_unclear_gap           360
disputed_territory                64
Name: count, dtype: int64

By destination_category:
destination_category
non_sovereign_territory         1640
successor_missing_from_cepii    1256
microstate_unclear_gap          1112
political_non_reporter           344
disputed_territory                56
Name: count, dtype: int64


**Step D — apply real fixes (deferred, documented as a future reference).**

Of the six categories, only `successor_missing_from_cepii` (1.57% of stock) has
a real, concrete fix available: `dist_cepii.xls` is the *dyadic* CEPII file
and predates Serbia/Montenegro/Curacao/Sint Maarten/South Sudan, but CEPII's
companion **`geo_cepii`** file has per-country coordinates (capital lat/long)
that could be used to compute great-circle distance directly for these 5
countries, filling `dist`/`distw` without inventing anything.

**Not applied yet** — `geo_cepii` hasn't been sourced. Flagging this explicitly
so it isn't silently forgotten:

- [ ] TODO: download `geo_cepii` from https://www.cepii.fr/CEPII/en/bdd_modele/bdd_modele_item.asp?id=6
- [ ] TODO: compute great-circle distance for (SRB, MNE, CUW, SXM, SSD) pairs from lat/long
- [ ] TODO: backfill `dist`/`distw` in `gravity_df` for the 2,104 origin-side +
      1,256 destination-side rows currently flagged `successor_missing_from_cepii`
- [ ] TODO: `colony`/`comlang_off`/`comcol` still can't be filled this way (not
      geometric facts) — these stay documented gaps even after the distance fix

All other categories (`microstate_unclear_gap`, `political_non_reporter`,
`disputed_territory`, `non_sovereign_territory`) are being left as documented
limitations, not fixed — their combined volume (<1.5% each) doesn't justify
the effort/risk of sourcing external data or approximating values.

**Step E — final summary, tying Steps A–D together.**

Includes the one number that actually matters going into Step 7: how many rows
have *every* core gravity variable present, i.e. are truly regression-ready
without any listwise deletion surprises.

In [47]:
print("="*70)
print("DATA QUALITY SUMMARY -- Country Categorization Pass (Steps A-D)")
print("="*70)

print(f"\nTotal gravity_df rows: {len(gravity_df):,}")
print(f"Total migrant_stock represented: {gravity_df['migrant_stock'].sum():,.0f}")
print(f"\nRows flagged with a documented category: {flagged.sum():,} ({100*flagged.mean():.2f}%)")

print("\nPer-category impact:")
for cat, items in country_categories.items():
    codes = set(items.keys())
    mask = gravity_df['origin_iso3'].isin(codes) | gravity_df['destination_iso3'].isin(codes)
    cat_stock = gravity_df.loc[mask, 'migrant_stock'].sum()
    pct = 100 * cat_stock / total_stock
    print(f"  {cat:<32} {mask.sum():>6,} rows  {pct:6.3f}% of stock")

print("\nDecisions taken (full rationale in Step C markdown):")
print("  - dissolved_no_successor_code   : no action needed (0% impact, never in gravity_df)")
print("  - successor_missing_from_cepii  : documented gap, fix deferred (Step D TODO -- geo_cepii)")
print("  - microstate_unclear_gap        : documented, negligible")
print("  - political_non_reporter        : documented gap, fix deferred (out of scope for now)")
print("  - disputed_territory            : documented, negligible")
print("  - non_sovereign_territory       : documented, negligible")

core_vars = ['dist', 'colony', 'comlang_off', 'GDP_origin', 'GDP_dest']
complete_mask = gravity_df[core_vars].notna().all(axis=1)
print(f"\nRows with all core gravity variables present (regression-ready): "
      f"{complete_mask.sum():,} / {len(gravity_df):,} ({100*complete_mask.mean():.2f}%)")

DATA QUALITY SUMMARY -- Country Categorization Pass (Steps A-D)

Total gravity_df rows: 74,783
Total migrant_stock represented: 1,589,366,186

Rows flagged with a documented category: 7,208 (9.64%)

Per-category impact:
  dissolved_no_successor_code           0 rows   0.000% of stock
  successor_missing_from_cepii      2,104 rows   1.573% of stock
  microstate_unclear_gap            1,464 rows   0.032% of stock
  political_non_reporter              768 rows   0.678% of stock
  disputed_territory                  120 rows   0.089% of stock
  non_sovereign_territory           2,864 rows   0.348% of stock

Decisions taken (full rationale in Step C markdown):
  - dissolved_no_successor_code   : no action needed (0% impact, never in gravity_df)
  - successor_missing_from_cepii  : documented gap, fix deferred (Step D TODO -- geo_cepii)
  - microstate_unclear_gap        : documented, negligible
  - political_non_reporter        : documented gap, fix deferred (out of scope for now)
  - dispute

## 7. Model — PPML gravity regression

Poisson Pseudo-Maximum Likelihood chosen over log-linear OLS specifically
because ~7.7% of `migrant_stock` values are zero (Section on zero-handling,
decided above) — PPML models the level directly via a log *link*, so it
handles zero-valued outcomes natively without transforming them or dropping
any rows. Robust (HC1) standard errors used since Poisson's default
variance-equals-mean assumption doesn't hold for real migration data.

In [49]:
import numpy as np
import statsmodels.api as sm
import statsmodels.formula.api as smf

core_vars = ['migrant_stock', 'dist', 'colony', 'comlang_off', 'GDP_origin', 'GDP_dest']
model_df = gravity_df.dropna(subset=core_vars).copy()
print("Regression-ready rows:", len(model_df))

model_df['log_dist'] = np.log(model_df['dist'])
model_df['log_GDP_origin'] = np.log(model_df['GDP_origin'])
model_df['log_GDP_dest'] = np.log(model_df['GDP_dest'])

formula = 'migrant_stock ~ log_dist + colony + comlang_off + log_GDP_origin + log_GDP_dest'
ppml_model = smf.glm(formula=formula, data=model_df, family=sm.families.Poisson()).fit(cov_type='HC1')
print(ppml_model.summary())

Regression-ready rows: 67415
                 Generalized Linear Model Regression Results                  
Dep. Variable:          migrant_stock   No. Observations:                67415
Model:                            GLM   Df Residuals:                    67409
Model Family:                 Poisson   Df Model:                            5
Link Function:                    Log   Scale:                          1.0000
Method:                          IRLS   Log-Likelihood:            -2.5171e+09
Date:                Wed, 05 Aug 2026   Deviance:                   5.0336e+09
Time:                        20:10:51   Pearson chi2:                 2.27e+10
No. Iterations:                     7   Pseudo R-squ. (CS):              1.000
Covariance Type:                  HC1                                         
                     coef    std err          z      P>|z|      [0.025      0.975]
----------------------------------------------------------------------------------
Intercept      

In [50]:
gravity_df.head()

,destination_iso3,origin_iso3,year,migrant_stock,wb_lookup_year,contig,comlang_off,comlang_ethno,colony,comcol,curcol,col45,smctry,dist,distcap,distw,distwces,Country_origin_x,Region_origin,SubRegion_origin,SurfAreaSqKm_origin,PopTotal_origin,PopDens_origin,PopGrowth%_origin,GDP_origin,...,GNI/CapAtlas_dest,GNIAtlas_dest,Imports%GDP_dest,IndValAdd%GDP_dest,InflConsPric%_dest,LifeExpBirth_dest,MerchTrade%GDP_dest,MobileSubs/100_dest,MortRateU5_dest,NetMigr_dest,UrbanPopGrowth%_dest,Country_origin_y,GrossCapForm%GDP_origin,MilExp%GDP_origin,RevenueExGrants%GDP_origin,SchEnrollPrim%_origin,TaxRevenue%GDP_origin,Country_dest_y,GrossCapForm%GDP_dest,MilExp%GDP_dest,RevenueExGrants%GDP_dest,SchEnrollPrim%_dest,TaxRevenue%GDP_dest,origin_category,destination_category
0,BDI,KEN,1990,186.0,2000.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,867.4281,867.4281,877.5054,852.7606,Kenya,Africa,Sub-Saharan Africa,580370.0,30851606.0,54.207411,2.915447,1.270535e+10,...,140.0,8.593409e+08,16.234754,15.539571,38.944893,47.514,22.745913,0.258733,154.6,-40114.0,4.621538,Kenya,17.414091,1.313742,NaN,94.130013,NaN,Burundi,2.781138,4.862402,NaN,60.719742,NaN,NaN,NaN
1,BDI,RWA,1990,221943.0,2000.0,1.0,1.0,1.0,0.0,1.0,0.0,0.0,1.0,180.0060,180.0060,162.1818,146.9860,Rwanda,Africa,Sub-Saharan Africa,26340.0,8109989.0,328.738914,1.245731,2.068763e+09,...,140.0,8.593409e+08,16.234754,15.539571,38.944893,47.514,22.745913,0.258733,154.6,-40114.0,4.621538,Rwanda,12.292775,3.535503,NaN,97.399910,NaN,Burundi,2.781138,4.862402,NaN,60.719742,NaN,NaN,NaN
2,BDI,UGA,1990,1833.0,2000.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,549.2128,549.2128,555.0759,526.8893,Uganda,Africa,Sub-Saharan Africa,241550.0,24020697.0,120.217692,3.135356,6.193247e+09,...,140.0,8.593409e+08,16.234754,15.539571,38.944893,47.514,22.745913,0.258733,154.6,-40114.0,4.621538,Uganda,19.483822,1.764608,NaN,128.961121,NaN,Burundi,2.781138,4.862402,NaN,60.719742,NaN,NaN,NaN
3,BDI,TZA,1990,11912.0,2000.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,1172.4890,777.6951,893.7339,687.5458,Tanzania,Africa,Sub-Saharan Africa,947300.0,34463704.0,38.906868,2.836808,1.337598e+10,...,140.0,8.593409e+08,16.234754,15.539571,38.944893,47.514,22.745913,0.258733,154.6,-40114.0,4.621538,Tanzania,17.456793,1.487264,NaN,68.970528,NaN,Burundi,2.781138,4.862402,NaN,60.719742,NaN,NaN,NaN
4,BDI,COD,1990,36654.0,2000.0,1.0,1.0,1.0,0.0,1.0,0.0,0.0,0.0,1559.9810,1559.9810,1209.6550,666.6916,"Congo, Dem. Rep.",Africa,Sub-Saharan Africa,2344860.0,48616317.0,21.444748,2.898841,1.908805e+10,...,140.0,8.593409e+08,16.234754,15.539571,38.944893,47.514,22.745913,0.258733,154.6,-40114.0,4.621538,"Congo, Dem. Rep.",14.433496,0.220261,NaN,NaN,NaN,Burundi,2.781138,4.862402,NaN,60.719742,NaN,NaN,NaN


In [51]:
gravity_df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 74783 entries, 0 to 74782
Data columns (total 81 columns):
 #   Column                      Non-Null Count  Dtype  
---  ------                      --------------  -----  
 0   destination_iso3            74751 non-null  object 
 1   origin_iso3                 74623 non-null  object 
 2   year                        74783 non-null  int32  
 3   migrant_stock               74783 non-null  float64
 4   wb_lookup_year              74783 non-null  float64
 5   contig                      70551 non-null  float64
 6   comlang_off                 70551 non-null  float64
 7   comlang_ethno               70551 non-null  float64
 8   colony                      70551 non-null  float64
 9   comcol                      70551 non-null  float64
 10  curcol                      70551 non-null  float64
 11  col45                       70551 non-null  float64
 12  smctry                      70551 non-null  float64
 13  dist                        705

In [52]:
model_df.info()

<class 'pandas.core.frame.DataFrame'>
Index: 67415 entries, 0 to 74777
Data columns (total 84 columns):
 #   Column                      Non-Null Count  Dtype  
---  ------                      --------------  -----  
 0   destination_iso3            67415 non-null  object 
 1   origin_iso3                 67415 non-null  object 
 2   year                        67415 non-null  int32  
 3   migrant_stock               67415 non-null  float64
 4   wb_lookup_year              67415 non-null  float64
 5   contig                      67415 non-null  float64
 6   comlang_off                 67415 non-null  float64
 7   comlang_ethno               67415 non-null  float64
 8   colony                      67415 non-null  float64
 9   comcol                      67415 non-null  float64
 10  curcol                      67415 non-null  float64
 11  col45                       67415 non-null  float64
 12  smctry                      67415 non-null  float64
 13  dist                        67415 no

## 9. Notebook closed here — decisions summary carried forward

This notebook's job (data sourcing, wrangling, merging, country-categorization,
and a working PPML gravity model) is done. Future analysis (VIF/multicollinearity
checks, overdispersion diagnostics, additional model specs, visualization) continues
in a fresh notebook, loading the exported `gravity_df.csv` below.

**Key decisions made in this notebook, carried forward so the next notebook isn't
context-free:**

1. **UN country filter**: `Location code < 900` separates real countries from
   region/aggregate rows (UN M49 convention).
2. **ISO3 crosswalk**: `pycountry` auto-match + documented manual overrides for
   ~20 World Bank names and ~15 UN names (long-form/historical naming).
3. **CEPII legacy codes fixed**: `ROM`→`ROU`, `ZAR`→`COD`, `TMP`→`TLS`, `PAL`→`PSE`.
   `YUG`/`ANT` left unmapped — genuinely obsolete entities, no current equivalent.
4. **Time-period alignment**: World Bank annual data snapped to the *nearest*
   available year for each UN 5-year snapshot (not exact-year, not interpolated) —
   1990/1995→2000, 2024→2021.
5. **Interpolated-file extras**: `IntermRegion` dropped (categorical, not a real
   gap) and `DomCredit%GDP` dropped (~71% of countries never report it — MNAR,
   not fillable). The other 5 extras kept but flagged as secondary-only
   (`RevenueExGrants%GDP`/`TaxRevenue%GDP` especially, ~38% missing).
6. **Country-categorization pass (Steps A–E)**: 6 categories of missing-country
   gaps identified and volume-weighted (dissolved states, CEPII-missing
   successor states, microstates, political non-reporters, disputed territory,
   non-sovereign territories). None imputed. One real fix deferred as an open
   TODO: sourcing `geo_cepii` to backfill distance for 5 successor states
   (~1.57% of migrant volume).
7. **`distw`/`distwces` dtype fixed** (Section 2c) — CEPII's `.` missing-value
   placeholder was forcing these columns to text; now proper `float64`.
8. **Zero-handling decision**: 7.7% of rows have `migrant_stock = 0` — these are
   real reported zero-flow corridors, not missing data. Confirmed *not*
   concentrated in a few countries (228/230 origins have at least one zero-row).
   **PPML chosen over log-linear OLS** specifically to handle these natively,
   without dropping rows or countries.
9. **First PPML model already run** (Section 7 above) on the 67,415-row
   regression-ready subset — `colony` (colonial tie) shows the single strongest
   effect, ahead of distance, language, and either country's GDP. **Not yet
   validated**: overdispersion diagnostic and VIF/multicollinearity check are
   the next real steps, to be done in the new notebook before trusting these
   coefficients as final.


In [54]:
# Export the FULL gravity_df (74,783 rows) -- not model_df (the 67,415-row
# regression-specific filtered copy). Keeping the full table lets a future
# notebook make its own filtering decisions for a different model spec,
# rather than being locked into today's core_vars choice.
gravity_df.to_csv('gravity_df.csv', index=False)
print(f"Exported gravity_df.csv -- {gravity_df.shape[0]:,} rows x {gravity_df.shape[1]} columns")


Exported gravity_df.csv -- 74,783 rows x 81 columns
